In [ ]:
from grid_intelligence.params import ENTSOE_API_KEY, EIA_API_KEY, DATA_DIR
from grid_intelligence.data.fetcher import EnergyDataFetcher
from datetime import datetime
import pandas as pd

fetcher = EnergyDataFetcher(
    entsoe_api_key=ENTSOE_API_KEY,
    eia_api_key=EIA_API_KEY,
    data_dir=DATA_DIR
)

today = datetime.now().strftime("%Y-%m-%d")
prices, load, gas = fetcher.fetch_all(start="2026-04-15", end=today)

df = prices.join(load, how='inner').join(gas, how='inner')
print(df.corr()['price'])

Fetching prices...
Prices gespeichert: (5661, 1)
Fetching load...


KeyboardInterrupt: 

In [55]:
print(prices.isnull().sum())
print(load.isnull().sum())
print(gas.isnull().sum())

print(prices.shape)
print(load.shape)
print(gas.shape)

price    0
dtype: int64
load    0
dtype: int64
WTI_Oil        0
Brent_Oil      0
Natural_Gas    0
ttf_gas        0
dtype: int64
(8637, 1)
(8633, 1)
(7873, 4)


In [52]:
# Prices - nur 1 Null, einfach interpolieren
prices = prices.interpolate(method='linear', limit=3)

# Load - schon sauber

# Gas - ffill für Wochenenden/Feiertage
gas = gas.ffill().bfill()

print(prices.isnull().sum())
print(gas.isnull().sum())

price    0
dtype: int64
WTI_Oil        0
Brent_Oil      0
Natural_Gas    0
ttf_gas        0
dtype: int64


In [29]:
df = prices.join(load, how='inner').join(gas, how='inner')
print(df.shape)
print(df.isnull().sum())
print(df.corr()['price'])

(18073, 6)
price          0
load           0
WTI_Oil        0
Brent_Oil      0
Natural_Gas    0
ttf_gas        0
dtype: int64
price          1.000000
load           0.338442
WTI_Oil       -0.078479
Brent_Oil     -0.049636
Natural_Gas    0.132922
ttf_gas        0.200117
Name: price, dtype: float64


In [45]:
import pandas as pd

prices = pd.read_csv("raw_data/raw_prices.csv", index_col=0, parse_dates=True)
load   = pd.read_csv("raw_data/raw_load.csv",   index_col=0, parse_dates=True)
gas    = pd.read_csv("raw_data/raw_gas.csv",     index_col=0, parse_dates=True)

print(prices.isnull().sum())
print(load.isnull().sum())
print(gas.isnull().sum())

price    1
dtype: int64
load    0
dtype: int64
WTI_Oil        0
Brent_Oil      0
Natural_Gas    0
ttf_gas        0
dtype: int64


In [46]:
gas = gas.ffill().bfill()
gas.to_csv("raw_data/raw_gas.csv")
print(gas.isnull().sum())

WTI_Oil        0
Brent_Oil      0
Natural_Gas    0
ttf_gas        0
dtype: int64


In [49]:
prices.index = pd.to_datetime(prices.index, utc=True)
load.index = pd.to_datetime(load.index, utc=True)
gas.index = pd.to_datetime(gas.index, utc=True)

print(pd.infer_freq(prices.index))
print(pd.infer_freq(load.index))
print(pd.infer_freq(gas.index))

h
h
h


In [53]:
import inspect
print(inspect.getsource(fetcher.fetch_load))

    def fetch_load(self, start: str, end: str, country: str = "DE_LU") -> pd.DataFrame:
        start_ts = pd.Timestamp(start, tz="Europe/Berlin")
        end_ts   = pd.Timestamp(end,   tz="Europe/Berlin")
        load = self.client.query_load(country, start=start_ts, end=end_ts)
        load = load.resample('h').mean()
        load.columns = ['load']
        return load



In [2]:
import sys
print(sys.executable)

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/bin/python
